# IOAI — 2024 First Stage Adversarial Attacks (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os, zipfile, urllib.request
if not os.path.exists('data/trained_model.pth'):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/2024-first-stage-adversarial-attacks/data.zip', 'd.zip')
    zipfile.ZipFile('d.zip').extractall('data')
print('데이터:', sorted(os.listdir('data')))
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# 적대적 공격 모범답안 (PGD, Projected Gradient Descent)

폴란드 AI 올림피아드 I · 2024 · 1단계. 분류기를 속이는 적대적 예제 생성. **PGD**: 분류기 손실을 키우는
방향(∇x)으로 이미지를 반복 이동하되, 원본과의 **L∞ 차이 ≤ 0.3** 로 사영(clip). 화이트박스 공격.

**성능(val 11000, 실측)**: base_acc 92% → 공격후 **0%**, 평균 SSIM 0.53 → criterion **48.4** → **100/100**.
(베이스라인=원본유지 0점.) *criterion=SSIM×정확도하락 이라, 하락을 최대로 하면 SSIM 0.5여도 42를 넘는다.*

**제출**: `submission.npz` — `perturbed(N,28,28)`.


In [ ]:
# 데이터 준비 (Colab: 자동 다운로드 / DGX: data/ 이미 존재)
import os, urllib.request, zipfile
if not os.path.exists("data/trained_model.pth"):
    url = "https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/2024-first-stage-adversarial-attacks/data.zip"
    urllib.request.urlretrieve(url, "d.zip"); zipfile.ZipFile("d.zip").extractall("data")

import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
device = "cuda" if torch.cuda.is_available() else "cpu"

# 속이려는 분류기 (고정)
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1,32,3,1,0); self.conv2 = nn.Conv2d(32,64,3,1,0)
        self.pool = nn.MaxPool2d(3,2); self.fc1 = nn.Linear(64*11*11,128); self.fc2 = nn.Linear(128,10)
    def forward(self, x):
        x = F.relu(self.conv1(x)); x = F.relu(self.pool(self.conv2(x))); x = torch.flatten(x,1)
        return F.log_softmax(self.fc2(self.fc1(x)), 1)

def normalize_samples(samples):   # 이미지별 [-1,1] 정규화 (원문제와 동일)
    s = samples.reshape(-1, 784).astype("float64")
    mn = s.min(1, keepdims=True); mx = s.max(1, keepdims=True)
    return ((2*(s-mn)/(mx-mn)) - 1).reshape(-1, 28, 28)

net = Net().to(device)
net.load_state_dict(torch.load("data/trained_model.pth", map_location=device)); net.eval()
X_validation = normalize_samples(np.load("data/contest_validation_samples.npy")/255.).astype("float32")
y_validation = np.load("data/contest_validation_labels.npy")
print("val", X_validation.shape, "| device", device)


In [ ]:
def perturbe_dataset(original_dataset, eps=0.3, alpha=0.02, steps=30):
    """PGD 공격: 원본 x0 에서 시작해 분류기 손실(정답에 대한 NLL)을 키우는 방향으로 반복 이동.
    매 스텝 부호경사 alpha 만큼 이동 후, x0 기준 L∞ 반경 eps 로 사영하고 [-1,1] 로 clip."""
    assert len(original_dataset.shape) == 3
    x0 = torch.tensor(original_dataset, dtype=torch.float32, device=device).unsqueeze(1)
    y = torch.tensor(y_validation, dtype=torch.long, device=device)
    x = x0.clone()
    for _ in range(steps):
        x.requires_grad_(True)
        loss = F.nll_loss(net(x), y)               # 손실을 키우면 오분류 유도
        grad, = torch.autograd.grad(loss, x)
        x = (x + alpha * grad.sign()).detach()
        x = torch.min(torch.max(x, x0 - eps), x0 + eps).clamp(-1, 1)   # L∞ 사영 + 범위 clip
    return x.squeeze(1).cpu().numpy()


In [ ]:
# val 공격 -> submission.npz
perturbed = perturbe_dataset(X_validation)
assert perturbed.shape == X_validation.shape
assert np.max(np.abs(perturbed - X_validation)) <= 0.3 + 1e-6, "L∞ 제약(0.3) 위반"
np.savez_compressed("submission.npz", perturbed=perturbed.astype("float32"))
print("submission.npz 저장:", perturbed.shape, "| L∞", round(float(np.max(np.abs(perturbed-X_validation))),3))


### 정리
- PGD(화이트박스, L∞≤0.3) 로 분류기를 92%→0% 로 붕괴 → criterion 48.4 → 100점.
- **핵심**: 손실 그래디언트의 부호 방향으로 최대 예산만큼 밀되, 원본 반경 안으로 사영해 지각 왜곡을 제한.


## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.npz']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)